# Section VI-A: Error Mitigation for Standard RB Circuits
**Figs. 18 & 19** — single-qubit and two-qubit EPC comparison with/without single-shot OTEM.

> Device and qubit selection are determined by preprocessing results.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # qatg package
sys.path.insert(0, os.path.abspath('.'))    # OTEM.py, Result_analyze.py

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from copy import deepcopy
from qiskit import QuantumCircuit, transpile, ClassicalRegister
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_experiments.library import StandardRB
from qiskit.result import marginal_counts
from otem import OTEM

## 1. Connect & preprocess

In [ ]:
service = QiskitRuntimeService(
    channel='ibm_cloud',
    instance='YOUR_IBMQ_INSTANCE_HERE',
    token='YOUR_IBMQ_TOKEN_HERE'
)
backend_name = 'ibm_kyoto'   # change to target device
backend = service.backend(backend_name)
nqb     = backend.num_qubits
print(f"Backend: {backend.name}  ({nqb} qubits)")

In [ ]:
shots = 10000
nqc   = 29
test_datas = [QuantumCircuit().from_qasm_file(f'test_circuits/test_circuit_{i}.qasm')
              for i in range(nqc)]
my_em  = OTEM(backend, test_datas)
sampler = Sampler(backend)
preprocess_qcs = my_em.build_preprocess_test()

job_pre = sampler.run(preprocess_qcs, shots=shots)
print('Preprocess job:', job_pre.job_id())
pre_result = job_pre.result()
_, _, _, test_id = my_em.qubit_and_online_test_selection_from_result(pre_result)
print('Selected test IDs:', {i: test_id[i] for i in range(nqb) if test_id[i] != -1})

In [ ]:
def em_exp_on_layouts(qc_alg, qc_ot, layouts, backend):
    qc = qc_ot.copy()
    crt = ClassicalRegister(len(layouts), name='testresult')
    qc.add_register(crt); qc.measure(range(len(layouts)), crt)
    for g in qc_alg:
        qc.append(g.operation, qargs=g.qubits)
    crm = ClassicalRegister(len(layouts), name='measureresult')
    qc.add_register(crm); qc.measure(range(len(layouts)), crm)
    return transpile(qc, backend, initial_layout=layouts, optimization_level=0)

def gen_ori_qc(qc, layouts, backend):
    qc_ori = qc.copy()
    crm = ClassicalRegister(len(layouts), name='measureresult')
    qc_ori.add_register(crm); qc_ori.measure(range(len(layouts)), crm)
    return transpile(qc_ori, backend, initial_layout=layouts, optimization_level=0)

def otem_get_marginal_counts_1qb(data, idx):
    tr = data.testresult.get_bitstrings()
    mr = data.measureresult.get_bitstrings()
    c = {}
    for t, m in zip(tr, mr):
        if t[idx] == '0':
            c[m[idx]] = c.get(m[idx], 0) + 1
    return c

def otem_get_marginal_counts_2qb(data, qb_ids):
    tr = data.testresult.get_bitstrings()
    mr = data.measureresult.get_bitstrings()
    c = {}
    for t, m in zip(tr, mr):
        if all(t[i] == '0' for i in qb_ids):
            key = ''.join(m[i] for i in qb_ids)
            c[key] = c.get(key, 0) + 1
    return c

def srate(counts, nqb=1):
    return counts.get('0'*nqb, 0) / sum(counts.values()) if counts else 0.0

def fit_rb(result_counts, nClif, nqb=1):
    d = 2**nqb
    p0 = np.array([srate(c, nqb) for c in result_counts])
    try:
        params, conv = curve_fit(lambda x,a,b,alpha: a*alpha**x+b,
                                  nClif, p0, [1-1/d, 1/d, 0.99])
    except Exception as e:
        print(e); return float('nan'), float('nan')
    a, b, alpha = params
    return (d-1)/d*(1-alpha), np.sqrt(conv[2][2])/2

def run_all(qcs, sampler, shots, max_circuits=30):
    results = []
    for i in range(0, len(qcs), max_circuits):
        job = sampler.run(qcs[i:i+max_circuits], shots=shots)
        print(f'  Submitted {i}–{min(i+max_circuits,len(qcs))}, job: {job.job_id()}')
        results.extend(list(job.result()))
    return results

def classify3(results):
    o,e,c = [],[],[]
    for i in range(len(results)//3):
        o.append(results[3*i]); e.append(results[3*i+1]); c.append(results[3*i+2])
    return o,e,c

## 2. Build single-qubit OT circuit

In [ ]:
def gen_ot_1qb(test_datas, test_id, nqb):
    qc, layouts = QuantumCircuit(0), []
    for i in range(nqb):
        if test_id[i] != -1:
            qc2 = test_datas[test_id[i]].copy(); qc2.remove_final_measurements()
            qc  = qc2.tensor(qc); layouts.append(i)
    return qc, layouts

qc_ot, layouts = gen_ot_1qb(test_datas, test_id, nqb)
print(f"Selected {len(layouts)} qubits for OTEM: {layouts}")

## 3. Single-qubit RB (Fig. 18)

In [ ]:
num_samples = 3
lengths_1qb = [10 + 50*i for i in range(10)]
qc_rb_1qb   = StandardRB([0], lengths_1qb, num_samples=num_samples).circuits()

def gen_para_1qb(qc, n):
    out = QuantumCircuit(0)
    for _ in range(n):
        q2 = qc.copy(); q2.remove_final_measurements(); out = q2.tensor(out)
    return out

qcs_rb = [gen_para_1qb(qc, len(layouts)) for qc in qc_rb_1qb]
qcs_ori  = [gen_ori_qc(qc, layouts, backend) for qc in qcs_rb]
qcs_em   = [em_exp_on_layouts(qc, qc_ot, layouts, backend) for qc in qcs_rb]
qcs_ctrl = [em_exp_on_layouts(qc, QuantumCircuit.copy_empty_like(qc_ot), layouts, backend) for qc in qcs_rb]
all_1qb  = [x for triple in zip(qcs_ori, qcs_em, qcs_ctrl) for x in triple]

job_1qb = sampler.run(all_1qb, shots=shots)
print('1QB RB job:', job_1qb.job_id())
res_1qb = list(job_1qb.result())

In [ ]:
ori_r, em_r, ctrl_r = classify3(res_1qb)
nClif_1 = sorted(lengths_1qb * num_samples)
r_1qb = []
for i in range(len(layouts)):
    c_ori  = [marginal_counts(r.data.measureresult.get_counts(), [i]) for r in ori_r]
    c_em   = [otem_get_marginal_counts_1qb(r.data, -(i+1)) for r in em_r]
    epc_o, _ = fit_rb(c_ori, nClif_1)
    epc_e, _ = fit_rb(c_em,  nClif_1)
    r_1qb.append((layouts[i], epc_o, epc_e))
    print(f"  Q{layouts[i]}: ori={epc_o:.5f}  em={epc_e:.5f}")

ori_e = [x[1] for x in r_1qb if not np.isnan(x[1])]
em_e  = [x[2] for x in r_1qb if not np.isnan(x[2])]
plt.figure(figsize=(5,5))
lim = max(max(ori_e), max(em_e)) * 1.1
plt.plot([0,lim],[0,lim],'k--',lw=1)
plt.scatter(ori_e, em_e, color='tab:red', s=40)
plt.xlabel('Original EPC'); plt.ylabel('Error mitigated EPC')
plt.title('Fig. 18: Single-qubit EPC')
plt.tight_layout()
plt.savefig('fig18_single_qubit_epc.pdf', bbox_inches='tight')
plt.show()
print(f"Avg reduction: {np.mean([(o-e)/o for o,e in zip(ori_e,em_e) if o>0])*100:.1f}%")

## 4. Two-qubit RB (Fig. 19)

In [ ]:
cmap = deepcopy(backend.coupling_map); cmap.make_symmetric()

def gen_ot_2qb(test_datas, test_id, nqb, cmap):
    qc, layouts = QuantumCircuit(0), []
    for i in range(nqb):
        if i not in layouts and test_id[i] != -1:
            for nb in cmap.neighbors(i):
                if nb not in layouts:
                    for q in [i, nb]:
                        q2 = test_datas[test_id[i]].copy(); q2.remove_final_measurements()
                        qc = q2.tensor(qc); layouts.append(q)
                    break
    return qc, layouts

qc_ot2, lay2 = gen_ot_2qb(test_datas, test_id, nqb, cmap)
lengths_2qb   = [5+15*i for i in range(10)]
qc_rb_2qb     = StandardRB([0,1], lengths_2qb, num_samples=num_samples).circuits()

def gen_para_2qb(qc, n_pairs):
    out = QuantumCircuit(0)
    q2 = qc.copy(); q2.remove_final_measurements()
    for _ in range(n_pairs): out = q2.tensor(out)
    return out

n_pairs = len(lay2)//2
qcs_rb2 = [gen_para_2qb(qc, n_pairs) for qc in qc_rb_2qb]
all_2qb = [x for triple in zip(
    [gen_ori_qc(q,lay2,backend) for q in qcs_rb2],
    [em_exp_on_layouts(q,qc_ot2,lay2,backend) for q in qcs_rb2],
    [em_exp_on_layouts(q,QuantumCircuit.copy_empty_like(qc_ot2),lay2,backend) for q in qcs_rb2])
    for x in triple]

job_2qb = sampler.run(all_2qb, shots=shots)
print('2QB RB job:', job_2qb.job_id())
res_2qb = list(job_2qb.result())

In [ ]:
ori_r2, em_r2, _ = classify3(res_2qb)
nClif_2 = sorted(lengths_2qb * num_samples)
r_2qb = []
for pi in range(n_pairs):
    i0, i1 = pi*2, pi*2+1
    c_ori = [marginal_counts(r.data.measureresult.get_counts(),[i0,i1]) for r in ori_r2]
    c_em  = [otem_get_marginal_counts_2qb(r.data,[-(i0+1),-(i1+1)]) for r in em_r2]
    epc_o, _ = fit_rb(c_ori, nClif_2, nqb=2)
    epc_e, _ = fit_rb(c_em,  nClif_2, nqb=2)
    r_2qb.append((lay2[i0], lay2[i1], epc_o, epc_e))
    print(f"  ({lay2[i0]},{lay2[i1]}): ori={epc_o:.5f}  em={epc_e:.5f}")

ori_2 = [x[2] for x in r_2qb if not np.isnan(x[2])]
em_2  = [x[3] for x in r_2qb if not np.isnan(x[3])]
plt.figure(figsize=(5,5))
lim2 = max(max(ori_2),max(em_2))*1.1
plt.plot([0,lim2],[0,lim2],'k--',lw=1)
plt.scatter(ori_2, em_2, color='tab:red', s=40)
plt.xlabel('Original EPC'); plt.ylabel('Error mitigated EPC')
plt.title('Fig. 19: Two-qubit EPC')
plt.tight_layout()
plt.savefig('fig19_two_qubit_epc.pdf', bbox_inches='tight')
plt.show()
print(f"Avg 2QB reduction: {np.mean([(o-e)/o for o,e in zip(ori_2,em_2) if o>0])*100:.1f}%")